# 🚀 Ollama & MCP Server Training Tutorial

Welcome to this comprehensive tutorial for Python developers who want to learn and practice with **Ollama** and **MCP (Model Context Protocol) servers**!

## What You'll Learn

- 🔧 Install and configure Ollama locally
- 📦 Set up the MCP Python SDK
- 🏗️ Build your own MCP server from scratch
- 🔌 Connect Ollama with your MCP server
- 🛠️ Implement advanced MCP features
- 🚀 Deploy and monitor your MCP server

## Prerequisites

- Python 3.8+ installed
- Basic understanding of Python programming
- Familiarity with async/await concepts (helpful but not required)

Let's get started! 🎯

## 1. 📥 Install and Setup Ollama

First, we need to install Ollama on your system. Ollama is a tool that allows you to run large language models locally.

### Installation Steps:
1. **Windows**: Download from [https://ollama.ai](https://ollama.ai)
2. **macOS**: `brew install ollama` or download from the website
3. **Linux**: `curl -fsSL https://ollama.ai/install.sh | sh`

### Verify Installation

In [ ]:
# Check if Ollama is running
import subprocess
import sys

def check_ollama_installation():
    """Check if Ollama is installed and running"""
    try:
        result = subprocess.run(['ollama', '--version'], 
                              capture_output=True, text=True, check=True)
        print("✅ Ollama is installed!")
        print(f"Version: {result.stdout.strip()}")
        return True
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("❌ Ollama is not installed or not in PATH")
        print("Please install Ollama from https://ollama.ai")
        return False

def check_ollama_service():
    """Check if Ollama service is running"""
    try:
        result = subprocess.run(['ollama', 'list'], 
                              capture_output=True, text=True, check=True)
        print("✅ Ollama service is running!")
        return True
    except subprocess.CalledProcessError:
        print("❌ Ollama service is not running")
        print("Please start Ollama service")
        return False

# Run checks
print("🔍 Checking Ollama installation...")
if check_ollama_installation():
    check_ollama_service()

In [ ]:
# Pull a small model for testing
import subprocess
import time

def pull_model(model_name="llama3.2:1b"):
    """Pull a model from Ollama registry"""
    print(f"📥 Pulling model: {model_name}")
    print("This may take a few minutes...")
    
    try:
        # Pull the model
        result = subprocess.run(['ollama', 'pull', model_name], 
                              capture_output=True, text=True, check=True)
        print(f"✅ Successfully pulled {model_name}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Error pulling model: {e}")
        return False

def test_model(model_name="llama3.2:1b", prompt="Say hello!"):
    """Test the model with a simple prompt"""
    print(f"🧪 Testing model: {model_name}")
    
    try:
        result = subprocess.run(['ollama', 'run', model_name, prompt], 
                              capture_output=True, text=True, check=True)
        print(f"✅ Model response: {result.stdout.strip()}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Error testing model: {e}")
        return False

# Pull and test a small model
model_name = "llama3.2:1b"  # Small model good for training
if pull_model(model_name):
    time.sleep(2)  # Wait a moment for the model to be ready
    test_model(model_name, "Hello! Can you tell me what 2+2 equals?")

## 2. 📦 Install MCP Python SDK

Now let's install the MCP (Model Context Protocol) Python SDK and related dependencies. MCP allows AI models to securely connect with external data sources and tools.

### What is MCP?
- **Model Context Protocol**: A protocol for connecting AI models with external tools and data
- **Secure**: Provides controlled access to resources
- **Extensible**: Easy to add new tools and capabilities

In [ ]:
# Install MCP SDK and dependencies
import subprocess
import sys

def install_package(package_name):
    """Install a Python package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"✅ Successfully installed {package_name}")
        return True
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package_name}")
        return False

# Install required packages
packages = [
    "mcp>=1.0.0",
    "httpx>=0.27.0", 
    "ollama>=0.3.0",
    "pydantic>=2.0.0",
    "asyncio"
]

print("📦 Installing MCP SDK and dependencies...")
for package in packages:
    install_package(package)

print("\n🎉 Installation complete!")
print("Now we can start building our MCP server!")

## 3. 🏗️ Create a Basic MCP Server

Let's create our first MCP server! We'll start with a simple server that can communicate with Ollama and provide basic functionality.

### MCP Server Components:
- **Server**: The main server instance
- **Tools**: Functions that the AI model can call
- **Resources**: Data that the AI model can access
- **Transport**: How the server communicates (stdio, HTTP, etc.)

In [ ]:
# Create a basic MCP server
import asyncio
import logging
from typing import Any

from mcp.server.models import InitializationOptions
from mcp.server import NotificationOptions, Server
from mcp.server.stdio import stdio_server
from mcp.types import (
    CallToolRequestParams,
    CallToolResult,
    ListToolsResult,
    TextContent,
    Tool,
)

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("basic-mcp-server")

class BasicMCPServer:
    """A basic MCP server for learning"""
    
    def __init__(self):
        self.server = Server("basic-mcp-server")
        self.setup_handlers()
        
    def setup_handlers(self):
        """Setup the server handlers"""
        
        @self.server.list_tools()
        async def handle_list_tools() -> ListToolsResult:
            """List available tools"""
            return ListToolsResult(
                tools=[
                    Tool(
                        name="greet",
                        description="Say hello to someone",
                        inputSchema={
                            "type": "object",
                            "properties": {
                                "name": {
                                    "type": "string",
                                    "description": "Name of the person to greet",
                                },
                            },
                            "required": ["name"],
                        },
                    ),
                    Tool(
                        name="calculate",
                        description="Perform simple math calculations",
                        inputSchema={
                            "type": "object",
                            "properties": {
                                "operation": {
                                    "type": "string",
                                    "description": "Math operation (add, subtract, multiply, divide)",
                                },
                                "a": {"type": "number", "description": "First number"},
                                "b": {"type": "number", "description": "Second number"},
                            },
                            "required": ["operation", "a", "b"],
                        },
                    ),
                ]
            )

        @self.server.call_tool()
        async def handle_call_tool(
            name: str, arguments: dict[str, Any] | None
        ) -> CallToolResult:
            """Handle tool calls"""
            
            if arguments is None:
                arguments = {}
                
            try:
                if name == "greet":
                    return await self._greet(arguments)
                elif name == "calculate":
                    return await self._calculate(arguments)
                else:
                    raise ValueError(f"Unknown tool: {name}")
                    
            except Exception as e:
                logger.error(f"Error in tool {name}: {str(e)}")
                return CallToolResult(
                    content=[TextContent(type="text", text=f"Error: {str(e)}")]
                )

    async def _greet(self, arguments: dict[str, Any]) -> CallToolResult:
        """Greet someone"""
        name = arguments.get("name", "World")
        message = f"Hello, {name}! Welcome to our MCP server! 👋"
        
        return CallToolResult(
            content=[TextContent(type="text", text=message)]
        )

    async def _calculate(self, arguments: dict[str, Any]) -> CallToolResult:
        """Perform calculations"""
        operation = arguments.get("operation")
        a = arguments.get("a")
        b = arguments.get("b")
        
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return CallToolResult(
                    content=[TextContent(type="text", text="Error: Cannot divide by zero")]
                )
            result = a / b
        else:
            return CallToolResult(
                content=[TextContent(type="text", text=f"Error: Unknown operation '{operation}'")]
            )
        
        message = f"{a} {operation} {b} = {result}"
        return CallToolResult(
            content=[TextContent(type="text", text=message)]
        )

# Create server instance
print("🏗️ Creating basic MCP server...")
basic_server = BasicMCPServer()
print("✅ Basic MCP server created!")
print("The server has 2 tools: 'greet' and 'calculate'")

## 4. 🛠️ Implement MCP Tools and Resources

Now let's enhance our MCP server with Ollama-specific tools. We'll add tools that can interact with Ollama models directly.

### Ollama Integration Tools:
- **list_models**: Get available Ollama models
- **generate_text**: Generate text using Ollama
- **chat**: Have conversations with Ollama models
- **pull_model**: Download new models

In [ ]:
# Enhanced MCP server with Ollama integration
import ollama
from typing import List, Dict

class OllamaMCPServer:
    """MCP Server with Ollama integration"""
    
    def __init__(self):
        self.server = Server("ollama-mcp-server")
        self.ollama_client = ollama.Client()
        self.setup_handlers()
        
    def setup_handlers(self):
        """Setup enhanced handlers with Ollama tools"""
        
        @self.server.list_tools()
        async def handle_list_tools() -> ListToolsResult:
            """List all available tools including Ollama tools"""
            return ListToolsResult(
                tools=[
                    # Basic tools
                    Tool(
                        name="greet",
                        description="Say hello to someone",
                        inputSchema={
                            "type": "object",
                            "properties": {
                                "name": {"type": "string", "description": "Name to greet"},
                            },
                            "required": ["name"],
                        },
                    ),
                    # Ollama tools
                    Tool(
                        name="list_ollama_models",
                        description="List all available Ollama models",
                        inputSchema={"type": "object", "properties": {}},
                    ),
                    Tool(
                        name="generate_text",
                        description="Generate text using an Ollama model",
                        inputSchema={
                            "type": "object",
                            "properties": {
                                "model": {"type": "string", "description": "Ollama model name"},
                                "prompt": {"type": "string", "description": "Text prompt"},
                                "temperature": {"type": "number", "description": "Creativity level (0-1)"},
                            },
                            "required": ["model", "prompt"],
                        },
                    ),
                    Tool(
                        name="chat_with_ollama",
                        description="Have a conversation with an Ollama model",
                        inputSchema={
                            "type": "object",
                            "properties": {
                                "model": {"type": "string", "description": "Ollama model name"},
                                "message": {"type": "string", "description": "Your message"},
                                "system_prompt": {"type": "string", "description": "System instructions"},
                            },
                            "required": ["model", "message"],
                        },
                    ),
                ]
            )

        @self.server.call_tool()
        async def handle_call_tool(
            name: str, arguments: dict[str, Any] | None
        ) -> CallToolResult:
            """Handle tool calls"""
            
            if arguments is None:
                arguments = {}
                
            try:
                if name == "greet":
                    return await self._greet(arguments)
                elif name == "list_ollama_models":
                    return await self._list_ollama_models()
                elif name == "generate_text":
                    return await self._generate_text(arguments)
                elif name == "chat_with_ollama":
                    return await self._chat_with_ollama(arguments)
                else:
                    raise ValueError(f"Unknown tool: {name}")
                    
            except Exception as e:
                logger.error(f"Error in tool {name}: {str(e)}")
                return CallToolResult(
                    content=[TextContent(type="text", text=f"Error: {str(e)}")]
                )

    async def _greet(self, arguments: dict[str, Any]) -> CallToolResult:
        """Greet someone"""
        name = arguments.get("name", "World")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Hello, {name}! 🤖")]
        )

    async def _list_ollama_models(self) -> CallToolResult:
        """List available Ollama models"""
        try:
            models = self.ollama_client.list()
            model_list = []
            
            for model in models.get('models', []):
                model_info = f"• {model['name']} (Modified: {model.get('modified_at', 'Unknown')})"
                model_list.append(model_info)
            
            if model_list:
                content = "📋 Available Ollama Models:\n" + "\n".join(model_list)
            else:
                content = "No models found. Try pulling a model first!"
                
            return CallToolResult(
                content=[TextContent(type="text", text=content)]
            )
        except Exception as e:
            return CallToolResult(
                content=[TextContent(type="text", text=f"Error listing models: {str(e)}")]
            )

    async def _generate_text(self, arguments: dict[str, Any]) -> CallToolResult:
        """Generate text using Ollama"""
        model = arguments["model"]
        prompt = arguments["prompt"]
        temperature = arguments.get("temperature", 0.7)
        
        try:
            response = self.ollama_client.generate(
                model=model,
                prompt=prompt,
                options={"temperature": temperature}
            )
            
            generated_text = response.get('response', '')
            return CallToolResult(
                content=[TextContent(type="text", text=f"🤖 Generated: {generated_text}")]
            )
        except Exception as e:
            return CallToolResult(
                content=[TextContent(type="text", text=f"Error generating text: {str(e)}")]
            )

    async def _chat_with_ollama(self, arguments: dict[str, Any]) -> CallToolResult:
        """Chat with Ollama model"""
        model = arguments["model"]
        message = arguments["message"]
        system_prompt = arguments.get("system_prompt", "You are a helpful assistant.")
        
        try:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": message}
            ]
            
            response = self.ollama_client.chat(
                model=model,
                messages=messages
            )
            
            reply = response.get('message', {}).get('content', '')
            return CallToolResult(
                content=[TextContent(type="text", text=f"💬 Assistant: {reply}")]
            )
        except Exception as e:
            return CallToolResult(
                content=[TextContent(type="text", text=f"Error in chat: {str(e)}")]
            )

# Create enhanced server
print("🚀 Creating enhanced Ollama MCP server...")
ollama_server = OllamaMCPServer()
print("✅ Enhanced server created with Ollama integration!")
print("Available tools: greet, list_ollama_models, generate_text, chat_with_ollama")

## 5. 🔌 Connect Ollama to MCP Server

Now let's test our MCP server by simulating how it would work with Ollama. In a real setup, this would involve configuring Ollama to use our MCP server.

### Connection Flow:
1. **Start MCP Server**: Run our server in stdio mode
2. **Configure Client**: Set up a client to communicate with the server  
3. **Test Communication**: Send requests and receive responses

In [ ]:
# Test MCP Server functionality
class MCPServerTester:
    """Test our MCP server without running it as a separate process"""
    
    def __init__(self, server_instance):
        self.server = server_instance
        
    async def test_list_tools(self):
        """Test listing available tools"""
        print("🔧 Testing tool listing...")
        
        # Get the handler function
        list_tools_handler = None
        for handler in self.server.server._list_tools_handlers:
            list_tools_handler = handler
            break
            
        if list_tools_handler:
            result = await list_tools_handler()
            print(f"✅ Found {len(result.tools)} tools:")
            for tool in result.tools:
                print(f"   • {tool.name}: {tool.description}")
        else:
            print("❌ No list_tools handler found")
    
    async def test_tool_call(self, tool_name: str, arguments: dict):
        """Test calling a specific tool"""
        print(f"🔧 Testing tool call: {tool_name}")
        
        # Get the handler function
        call_tool_handler = None
        for handler in self.server.server._call_tool_handlers:
            call_tool_handler = handler
            break
            
        if call_tool_handler:
            result = await call_tool_handler(tool_name, arguments)
            print(f"✅ Tool response:")
            for content in result.content:
                print(f"   {content.text}")
        else:
            print("❌ No call_tool handler found")

# Test our enhanced server
async def run_server_tests():
    """Run comprehensive tests on our MCP server"""
    print("🧪 Starting MCP Server Tests\n")
    
    tester = MCPServerTester(ollama_server)
    
    # Test 1: List tools
    await tester.test_list_tools()
    print()
    
    # Test 2: Test greet tool
    await tester.test_tool_call("greet", {"name": "Python Developer"})
    print()
    
    # Test 3: Test Ollama model listing (may fail if Ollama not running)
    await tester.test_tool_call("list_ollama_models", {})
    print()
    
    # Test 4: Test text generation (may fail if Ollama not running)
    await tester.test_tool_call("generate_text", {
        "model": "llama3.2:1b",
        "prompt": "What is Python?",
        "temperature": 0.5
    })
    print()
    
    print("🎉 Server tests completed!")

# Run the tests
await run_server_tests()

## 6. 🏋️ Practice Exercises

Now it's time to practice! Try these exercises to deepen your understanding:

### Exercise 1: Add a New Tool
Create a new tool called `translate_text` that uses Ollama to translate text between languages.

### Exercise 2: File Operations  
Add tools to read and write files (safely, with proper validation).

### Exercise 3: Web Search
Create a tool that can search the web and return results.

### Exercise 4: Data Analysis
Build tools that can analyze CSV data and return insights.

In [ ]:
# Exercise Template: Add your own tool!
class MyCustomMCPServer(OllamaMCPServer):
    """Extend the Ollama MCP server with your own tools"""
    
    def setup_handlers(self):
        """Setup handlers including your custom tools"""
        # Call parent setup first
        super().setup_handlers()
        
        # Add your custom tool here
        # Example: Weather tool, Calculator, File reader, etc.
        
        # TODO: Implement your custom tool
        pass
    
    async def _my_custom_tool(self, arguments: dict[str, Any]) -> CallToolResult:
        """Your custom tool implementation"""
        # TODO: Implement your tool logic here
        return CallToolResult(
            content=[TextContent(type="text", text="Custom tool result")]
        )

print("🎯 Template ready for your custom tools!")
print("💡 Ideas for tools:")
print("   • Weather checker")
print("   • Code formatter") 
print("   • URL shortener")
print("   • Password generator")
print("   • JSON validator")
print("\n📝 Edit the class above to add your own tools!")

## 7. 🎓 Next Steps & Resources

Congratulations! You've learned the basics of Ollama and MCP servers. Here's what to explore next:

### 🚀 Advanced Topics
- **Production Deployment**: Deploy your MCP server with proper logging and monitoring
- **Security**: Implement authentication and input validation
- **Performance**: Add caching and optimize for scale
- **Integration**: Connect with databases, APIs, and other services

### 📚 Resources
- **MCP Documentation**: [https://modelcontextprotocol.io](https://modelcontextprotocol.io)
- **Ollama Docs**: [https://ollama.ai/docs](https://ollama.ai/docs)
- **Python MCP SDK**: [https://github.com/modelcontextprotocol/python-sdk](https://github.com/modelcontextprotocol/python-sdk)
- **Examples**: Check the `examples/` folder in this project

### 💡 Project Ideas
1. **Personal Assistant**: Build an MCP server that manages your calendar, emails, and tasks
2. **Code Helper**: Create tools for code formatting, testing, and documentation
3. **Data Pipeline**: Build tools for data processing and analysis
4. **API Gateway**: Create an MCP server that interfaces with multiple APIs

### 🤝 Community
- Join the MCP community discussions
- Share your MCP servers on GitHub
- Contribute to the MCP ecosystem

**Happy coding! 🎉**